[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/agents/google-genai/sequential-agent.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>

# Google GenAI SDK - 构建顺序智能体

## 安装依赖

In [ ]:
!pip install -q google-genai arize-phoenix-otel openinference-instrumentation-google-genai

## 连接 Arize Phoenix

In [ ]:
import os
from getpass import getpass

from google import genai
from phoenix.otel import register

if "PHOENIX_API_KEY" not in os.environ:
    os.environ["PHOENIX_API_KEY"] = getpass("🔑 请输入 Phoenix API 密钥： ")

if "PHOENIX_COLLECTOR_ENDPOINT" not in os.environ:
    os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = getpass("🔑 请输入 Phoenix 收集器端点： ")

tracer_provider = register(auto_instrument=True, project_name="google-genai-sequential-agent")
tracer = tracer_provider.get_tracer(__name__)

## 通过 Google Vertex AI 进行身份验证

In [ ]:
!gcloud auth login

In [ ]:
# 使用 Vertex AI API 创建客户端，此处也可以改用 Google GenAI API
client = genai.Client(vertexai=True, project="<填写 GCP 项目 ID>", location="us-central1")

# 顺序智能体

In [ ]:
# --- 1. 定义辅助方法 ---


def get_model(model_name):
    """实例化并返回模型。"""
    return model_name


@tracer.chain
def research_topic(client, model, user_input):
    """根据用户输入研究主题。"""
    research_prompt = (
        "你是研究助手。\n"
        "请全面概述以下主题。\n"
        "包括关键事实、历史背景和当前意义。\n"
        "回复控制在 3–4 段。\n"
        "用户输入： "
    )
    research_response = client.models.generate_content(
        model=model,
        contents=research_prompt + user_input,
    )
    return research_response.text.strip()


@tracer.chain
def identify_key_points(client, model, research):
    """从研究中提取要点。"""
    key_points_prompt = (
        "你是研究总结助手。\n"
        "从以下研究中提取 5–7 个要点。\n"
        "每个要点以项目符号列出，并简要说明。\n\n"
        f"研究：\n{research}"
    )
    key_points_response = client.models.generate_content(model=model, contents=key_points_prompt)
    return key_points_response.text.strip()


@tracer.chain
def generate_future_directions(client, model, research, key_points):
    """根据研究与要点生成后续研究方向。"""
    future_prompt = (
        "你是研究策略专家。\n"
        "根据所提供的研究和要点，提出 3–4 个有前景的后续研究方向。\n"
        "说明各方向的潜在意义和影响。\n\n"
        f"研究：\n{research}\n\n"
        f"要点：\n{key_points}\n\n"
    )
    future_response = client.models.generate_content(model=model, contents=future_prompt)
    return future_response.text.strip()


# --- 2. 主执行过程 ---
@tracer.agent()
def run_agent(user_input):
    # 实例化模型
    research_model = get_model("gemini-2.0-flash-001")

    # 第 1 步：生成初始研究
    initial_research = research_topic(client, research_model, user_input)

    # 第 2 步：提取研究要点
    extracted_key_points = identify_key_points(client, research_model, initial_research)

    # 第 3 步：生成后续研究方向
    future_directions = generate_future_directions(
        client, research_model, initial_research, extracted_key_points
    )

    # 显示结果
    print("=== 初始研究 ===\n")
    print(initial_research)
    print("\n=== 要点 ===\n")
    print(extracted_key_points)
    print("\n=== 后续研究方向 ===\n")
    print(future_directions)

    return {
        "research": initial_research,
        "key_points": extracted_key_points,
        "future_directions": future_directions,
    }


run_agent(user_input=input("请输入希望研究的主题： "))